In [13]:
import os
import gymnasium as gym
import panda_gym

from huggingface_sb3 import load_from_hub, package_to_hub
from stable_baselines3 import A2C
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecVideoRecorder
from stable_baselines3.common.env_util import make_vec_env
from huggingface_hub import notebook_login

In [6]:
env_id = "PandaReachDense-v3"

env = gym.make(env_id)

s_size = env.observation_space.shape
a_size = env.action_space

s_size, a_size

rand_s = env.observation_space.sample
rand_a = env.action_space.sample

rand_s, rand_a

argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886


(<bound method Dict.sample of Dict('achieved_goal': Box(-10.0, 10.0, (3,), float32), 'desired_goal': Box(-10.0, 10.0, (3,), float32), 'observation': Box(-10.0, 10.0, (6,), float32))>,
 <bound method Box.sample of Box(-1.0, 1.0, (3,), float32)>)

In [7]:
env = make_vec_env(env_id, n_envs = 4)

env = VecNormalize(
    venv = env,
    norm_reward = True,
    norm_obs = True,
    clip_obs = 10.
)

argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886


In [8]:
model = A2C(
    policy = "MultiInputPolicy",
    env = env,
    learning_rate = 0.0007,
    n_steps = 5,
    gamma = 0.99,
    verbose = 1
)

Using cuda device


In [9]:
model.learn(1e6)

------------------------------------
| rollout/              |          |
|    ep_len_mean        | 41.2     |
|    ep_rew_mean        | -11.1    |
|    success_rate       | 0.2      |
| time/                 |          |
|    fps                | 241      |
|    iterations         | 100      |
|    time_elapsed       | 8        |
|    total_timesteps    | 2000     |
| train/                |          |
|    entropy_loss       | -4.22    |
|    explained_variance | 0.839    |
|    learning_rate      | 0.0007   |
|    n_updates          | 99       |
|    policy_loss        | -1.06    |
|    std                | 0.989    |
|    value_loss         | 0.37     |
------------------------------------
------------------------------------
| rollout/              |          |
|    ep_len_mean        | 41.3     |
|    ep_rew_mean        | -11.6    |
|    success_rate       | 0.213    |
| time/                 |          |
|    fps                | 272      |
|    iterations         | 200      |
|

In [10]:
model.save("PandaReachDense-v3")
env.save("vec_normalize.pkl")

In [4]:
eval_env = DummyVecEnv([lambda: gym.make("PandaReachDense-v3")])
eval_env = VecNormalize.load("vec_normalize.pkl", eval_env)

eval_env.render_mode = "rgb_array"

eval_env.training = False

eval_env.norm_reward = False

path = "models/PandaReachDense-v3"
model = A2C.load(path)

mean, std = evaluate_policy(model, eval_env)

mean, std

argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886


/mnt/c/Users/abhit/OneDrive/Documents/Programming/neuron/GOD1/lib/python3.14/site-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


(np.float64(-0.24381887121126056), np.float64(0.12936481631457158))

In [ ]:
notebook_login()

In [ ]:
package_to_hub(
    model = model,
    model_name = "PandaReachDense",
    model_architecture = "A2C",
    env_id = env_id,
    eval_env = eval_env,
    repo_id = #"your repo id",
    commit_message = "Commited"
)

In [ ]:
def record_sb3_video(env_id, model, video_length=1000, video_folder="videos"):
    """Records an evaluation video of a Stable Baselines3 model using its built-in

    VecVideoRecorder.
    """
    os.makedirs(video_folder, exist_ok=True)

    # 1. Create a vectorized environment with the right render mode
    # Note: If your assignment requires old 'gym', swap 'gymnasium' to 'gym' here
    eval_env = DummyVecEnv(
        [lambda: gym.make(env_id, render_mode="rgb_array")]
    )

    # 2. Wrap it with the SB3 Video Recorder
    # This automatically intercepts environment steps and records frames
    eval_env = VecVideoRecorder(
        eval_env,
        video_folder=video_folder,
        record_video_trigger=lambda step: step == 0,  # Start recording at step 0
        video_length=video_length,  # Record for this many steps
        name_prefix=f"sb3-{env_id}",
    )

    print(f"🎬 Recording evaluation video for {env_id}...")

    # 3. Standard SB3 Environment Loop
    obs = eval_env.reset()
    for _ in range(video_length):
        # SB3 uses model.predict instead of your old custom policy.act
        action, _states = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = eval_env.step(action)

    # 4. Close the environment to force moviepy to finish writing the file
    eval_env.close()
    print(f"🎉 Done! Video successfully saved to the '{video_folder}' folder.")

from stable_baselines3 import A2C

# Assuming you already defined and trained your model:
# model = A2C("MlpPolicy", "CartPole-v1", verbose=1).learn(total_timesteps=10000)

# Run the recorder
record_sb3_video(env_id= env_id , model=model, video_length=500)